In [1]:
using PyPlot
using LinearAlgebra
using SparseArrays
using Statistics

# The linear and nonlinear (elastic) wave equation in 2D

The linear wave equation:
$$
\begin{cases}
  \partial_t\,\vec u \,+\,\nabla\cdot\,G\,&=&\,0,
  \\[0.3em]
  \partial_t\,G \,+k_{\text{lin}}\big(\nabla\vec u\big)\,&=&\,0,
\end{cases}
$$
The nonlinear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,\vec u \,+\,\nabla\cdot\,G\,&=\,0,
  \\[0.3em]
  \partial_t\,G \,+\big(\mathbb{1}+\nabla\vec u\big)k\big(\nabla\vec u\big)\,&=\,0.
\end{cases}
$$
Here,
\begin{align*}
  k(\nabla\vec u) \,&:=\, \mathbb{C}:\Big\{\frac{1}{2}(\nabla\vec u+\nabla\vec u^T+\nabla\vec u^T\cdot\nabla\vec u)\Big\}
  \\[0.3em]
  k_{\text{lin}}(\nabla\vec u) \,&:=\, \mathbb{C}:\Big\{\frac{1}{2}(\nabla\vec u+\nabla\vec u^T)\Big\}
\end{align*}

## Boundary conditions:

Dirichlet conditions on $u$ correspond to a fixed displacement, i.e., the string/beam is attached to something:
$$
u(t,x) = 0.
$$

Neumann conditions on $u$ correspond to "stress free boundary conditions" (if we set the normal derivative to zero), or a "fixed stress" boundary condition that models a weight pulling on the string/beam:
$$
n\cdot\big(\mathbb{1}+\nabla\vec u\big)k\big(\nabla\vec u\big) = g(x).
$$

## Forcing terms:

A force $f(t,x)$ acts on the momentum equation, which reads (in second order form):
$$
\partial_t^2u + \Delta u \;=\; f(t,x)
$$
We can translate this into first order form by finding a function $F(t, x)$ with $\nabla\cdot F(t,x) = \vec f(t,x)$:

The linear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,\vec u \,+\,\nabla\cdot\,G\,&=&\,0,
  \\[0.3em]
  \partial_t\,G \,+k_{\text{lin}}\big(\nabla\vec u\big)\,&=&\,-F(t,x),
\end{cases}
$$
The nonlinear elasticity-wave equation:
$$
\begin{cases}
  \partial_t\,\vec u \,+\,\nabla\cdot\,G\,&=\,0,
  \\[0.3em]
  \partial_t\,G \,+\big(\mathbb{1}+\nabla\vec u\big)k\big(\nabla\vec u\big)\,&=\,-F(t,x).
\end{cases}
$$

# Space discretization

In [2]:
@enum BoundaryType do_nothing dirichlet_u dirichlet_u_normal dirichlet_g_normal periodic

struct Discretization
    Ω::Tuple{Tuple{Float64, Float64}, Tuple{Float64, Float64}}
    h::Tuple{Float64, Float64}
    N::Tuple{Int64, Int64}
    left_boundary::BoundaryType
    right_boundary::BoundaryType
    top_boundary::BoundaryType
    bottom_boundary::BoundaryType
    filter_mean_G::Bool
    stabilization::Float64
end

In [3]:
function MN(Ω, h)
    M, N = (Ω[2] .- Ω[1]) ./ h
    @assert(abs((M - floor(M)) * (M - ceil(M))) < 1.0e-12 && abs((N - floor(N)) * (N - ceil(N))) < 1.0e-12)
    # We index from 1 onwards
    return (round(Int64, M) + 1, round(Int64, N) + 1)
end

MN (generic function with 1 method)

In [4]:
function position(discretization, (i, j))
    @assert(i >= 1 && j >= 1)
    @assert(i <= discretization.N[1] && j <= discretization.N[2])
    Ω = discretization.Ω
    h = discretization.h
    return Ω[1] .+ h .* (i - 1, j - 1)
end

position (generic function with 1 method)

# Finite Difference Scheme

In [5]:
function gradient_interior(discretization, (i, j), U::Matrix{NTuple{2, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    
    @assert(i > 1 && i < M && j > 1 && j < N)

    U_x = 1. / (2. * h_x) .* (U[i + 1, j] .- U[i - 1, j])
    U_y = 1. / (2. * h_y) .* (U[i, j + 1] .- U[i, j - 1])
    
    return (U_x[1], U_y[1], U_x[2], U_y[2])
end

gradient_interior (generic function with 1 method)

In [6]:
function gradient(discretization, (i, j), U::Matrix{NTuple{2, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    
    U_x = (0., 0.)
    if (i == 1 && discretization.left_boundary != periodic)
        U_x = 1. / h_x .* (U[i + 1, j] .- U[i, j])
    elseif (i == 1 && discretization.left_boundary == periodic)
        U_x = 1. / (2. * h_x) .* (U[i + 1, j] .- U[M - 1, j]) #!
    elseif (i == M && discretization.right_boundary != periodic)
        U_x = 1. / h_x .* (U[i, j] .- U[i - 1, j])
    elseif (i == M && discretization.right_boundary == periodic)
        U_x = 1. / (2. * h_x) .* (U[1 + 1, j] .- U[i - 1, j]) #! 
    else
        U_x = 1. / (2. * h_x) .* (U[i + 1, j] .- U[i - 1, j])
    end

    U_y = (0., 0.)
    if (j == 1 && discretization.bottom_boundary != periodic)
        U_y = 1. / h_y .* (U[i, j + 1] .- U[i, j])
    elseif (j == 1 && discretization.bottom_boundary == periodic)
        U_y = 1. / (2. * h_y) .* (U[i, j + 1] .- U[i, N - 1]) #!
    elseif (j == N && discretization.top_boundary != periodic)
        U_y = 1. / h_y .* (U[i, j] .- U[i, j - 1])
    elseif (j == N && discretization.top_boundary == periodic)
        U_y = 1. / (2. * h_y) .* (U[i, 1 + 1] .- U[i, j - 1]) #!     
    else
        U_y = 1. / (2. * h_y) .* (U[i, j + 1] .- U[i, j - 1])
    end
    
    return (U_x[1], U_y[1], U_x[2], U_y[2])
end

gradient (generic function with 1 method)

In [7]:
function divergence_interior(discretization, (i, j), G::Matrix{NTuple{4, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    
    @assert(i > 1 && i < M && j > 1 && j < N)
    
    G_x = 1. / (2. * h_x) .* (G[i + 1, j] .- G[i - 1, j])
    G_y = 1. / (2. * h_y) .* (G[i, j + 1] .- G[i, j - 1])

    return(G_x[1] + G_y[2], G_x[3] + G_y[4])
end

divergence_interior (generic function with 1 method)

In [8]:
function divergence(discretization, (i, j), G::Matrix{NTuple{4, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    
    G_x = (0., 0., 0., 0.)
    if (i == 1 && discretization.left_boundary != periodic)
        G_x = 1. / h_x .* (G[i + 1, j] .- G[i, j])
    elseif (i == 1 && discretization.left_boundary == periodic)
        G_x = 1. / (2. * h_x) .* (G[i + 1, j] .- G[M - 1, j]) #!
    elseif (i == M && discretization.right_boundary != periodic)
        G_x = 1. / h_x .* (G[i, j] .- G[i - 1, j])
    elseif (i == M && discretization.right_boundary == periodic)
        G_x = 1. / (2. * h_x) .* (G[1 + 1, j] .- G[i - 1, j]) #! 
    else
        G_x = 1. / (2. * h_x) .* (G[i + 1, j] .- G[i - 1, j])
    end

    G_y = (0., 0., 0., 0.)
    if (j == 1 && discretization.bottom_boundary != periodic)
        G_y = 1. / h_y .* (G[i, j + 1] .- G[i, j])
    elseif (j == 1 && discretization.bottom_boundary == periodic)
        G_y = 1. / (2. * h_y) .* (G[i, j + 1] .- G[i, N - 1]) #!
    elseif (j == N && discretization.top_boundary != periodic)
        G_y = 1. / h_y .* (G[i, j] .- G[i, j - 1])
    elseif (j == N && discretization.top_boundary == periodic)
        G_y = 1. / (2. * h_y) .* (G[i, 1 + 1] .- G[i, j - 1]) #!     
    else
        G_y = 1. / (2. * h_y) .* (G[i, j + 1] .- G[i, j - 1])
    end
    
    return(G_x[1] + G_y[3], G_x[2] + G_y[4]) #DEBUG
end

divergence (generic function with 1 method)

In [9]:
function stabilization_term_interior(discretization, (i, j), U::Matrix{NTuple{2, Float64}}, G::Matrix{NTuple{4, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    stabilization = discretization.stabilization
    
    @assert(i > 1 && i < M && j > 1 && j < N)

    U_xx = stabilization .* (U[i + 1, j] .- 2. .* U[i, j] .+ U[i - 1, j])
    U_yy = stabilization .* (U[i, j + 1] .- 2. .* U[i, j] .+ U[i, j - 1])

    G_xx = stabilization .* (G[i + 1, j] .- 2. .* G[i, j] .+ G[i - 1, j])
    G_yy = stabilization .* (G[i, j + 1] .- 2. .* G[i, j] .+ G[i, j - 1])
    
    return (U_xx .+ U_yy, G_xx .+ G_yy)
end

stabilization_term_interior (generic function with 1 method)

In [10]:
function stabilization_term(discretization, (i, j), U::Matrix{NTuple{2, Float64}}, G::Matrix{NTuple{4, Float64}})
    M, N = discretization.N
    h_x, h_y = discretization.h
    stabilization = discretization.stabilization

    U_xx = (0., 0.)
    G_xx = (0., 0., 0., 0.)
    if (i == 1 && discretization.left_boundary != periodic)
        # no stabilization
    elseif (i == 1 && discretization.left_boundary == periodic)
        U_xx = stabilization .* (U[i + 1, j] .- 2. .* U[i, j] .+ U[M - 1, j])
        G_xx = stabilization .* (G[i + 1, j] .- 2. .* G[i, j] .+ G[M - 1, j])
    elseif (i == M && discretization.right_boundary != periodic)
        # no stabilization
    elseif (i == M && discretization.right_boundary == periodic) 
        U_xx = stabilization .* (U[1 + 1, j] .- 2. .* U[i, j] .+ U[i - 1, j])
        G_xx = stabilization .* (G[1 + 1, j] .- 2. .* G[i, j] .+ G[i - 1, j])
    else
        U_xx = stabilization .* (U[i + 1, j] .- 2. .* U[i, j] .+ U[i - 1, j])
        G_xx = stabilization .* (G[i + 1, j] .- 2. .* G[i, j] .+ G[i - 1, j])
    end

    U_yy = (0., 0.)
    G_yy = (0., 0., 0., 0.)
    if (j == 1 && discretization.bottom_boundary != periodic)
        # no stabilization
    elseif (j == 1 && discretization.bottom_boundary == periodic)
         U_yy = stabilization .* (U[i, j + 1] .- 2. .* U[i, j] .+ U[i, N - 1])
         G_yy = stabilization .* (G[i, j + 1] .- 2. .* G[i, j] .+ G[i, N - 1])
    elseif (j == N && discretization.top_boundary != periodic)
        # no stabilization
    elseif (j == N && discretization.top_boundary == periodic)
         U_yy = stabilization .* (U[i, 1 + 1] .- 2. .* U[i, j] .+ U[i, j - 1])
         G_yy = stabilization .* (G[i, 1 + 1] .- 2. .* G[i, j] .+ G[i, j - 1])
    else
         U_yy = stabilization .* (U[i, j + 1] .- 2. .* U[i, j] .+ U[i, j - 1])
         G_yy = stabilization .* (G[i, j + 1] .- 2. .* G[i, j] .+ G[i, j - 1])
    end
    
    return (U_xx .+ U_yy, G_xx .+ G_yy)
end

stabilization_term (generic function with 1 method)

In [11]:
function step(discretization, cfl, k, F, U_old, G_old, t_old, U_new, G_new)
    M, N = discretization.N
    h_x, h_y = discretization.h
    τ = cfl(min(h[1], h[2]))

    #
    # Central difference for interior degrees of freedom with stabilization:
    #

    for i in 2:M-1
        for j in 2:N-1
            F_ij = F(t_old, position(discretization, (i, j)))
            U_stab, G_stab = stabilization_term_interior(discretization, (i, j), U_old, G_old)
            U_new[i, j] = U_old[i, j] .- τ .* divergence_interior(discretization, (i, j), G_old) .+ τ .* U_stab
            U_prime = gradient_interior(discretization, (i, j), U_old)
            G_new[i, j] = G_old[i, j] .- τ .* k(U_prime) .- τ .* F_ij .+ τ .* G_stab
        end
    end

    #
    # One-sided difference operator in normal direction on the boundary:
    #

    boundary_update(i, j) = begin
        F_ij = F(t_old, position(discretization, (i, j)))
        U_stab, G_stab = stabilization_term(discretization, (i, j), U_old, G_old)
        U_new[i, j] = U_old[i, j] .- τ .* divergence(discretization, (i, j), G_old) .+ τ .* U_stab
        U_prime = gradient(discretization, (i, j), U_old)
        G_new[i, j] = G_old[i, j] .- τ .* k(U_prime) .- τ .* F_ij .+ τ .* G_stab
    end

    for i in 1:M
        for j in (1, N)
            boundary_update(i, j)
        end
    end

    for i in (1, M)
        for j in 1:N
            boundary_update(i, j)
        end
    end

    #
    # Post-process boundary values:
    #

    for (i, id) = ((1, discretization.left_boundary), (M, discretization.right_boundary))
        if id == dirichlet_u
            for j in 1:N
                U_new[i, j] = U_analytic(t_old + τ, position(discretization, (i, j)))
            end
        elseif id == dirichlet_u_normal
            for j in 1:N
                U_new[i, j] = (U_analytic(t_old + τ, position(discretization, (i, j)))[1], U_new[i, j][2])                
            end            
        elseif id == dirichlet_g_normal
            for j in 1:N
                F_ij = F(t_old + τ, position(discretization, (i, j)))
                U_stab, G_stab = stabilization_term(discretization, (i, j), U_old, G_old)
                g_ij = g(t_old + τ, position(discretization, (i, j)))
                direction = (i == 1) ? -1 : +1 # outward pointing normal
                G_new[i, j] = (
                    G_old[i, j][1] - direction * τ * g_ij[1] - τ * F_ij[1] + τ * G_stab[1], # G_11 selected by n = (1,0)
                    G_old[i, j][2] - direction * τ * g_ij[2] - τ * F_ij[2] + τ * G_stab[2], # G_12 selected by n = (1,0)
                    G_new[i, j][3],
                    G_new[i, j][4])
            end
        elseif id == periodic
            # do nothing
        else
            @assert(id == do_nothing, "not implemented")
        end
    end
    
    for (j, id) = ((1, discretization.top_boundary), (N, discretization.bottom_boundary))
        if id == dirichlet_u
            for i in 1:M
                U_new[i, j] = U_analytic(t_old + τ, position(discretization, (i, j)))
            end
        elseif id == dirichlet_u_normal
            for i in 1:M
                U_new[i, j] = (U_new[i, j][1], U_analytic(t_old + τ, position(discretization, (i, j)))[2])                
            end   
        elseif id == dirichlet_g_normal
            for i in 1:M
                F_ij = F(t_old, position(discretization, (i, j)))
                U_stab, G_stab = stabilization_term(discretization, (i, j), U_old, G_old)
                g_ij = g(t_old + τ, position(discretization, (i, j)))
                direction = (j == 1) ? -1 : +1 # outward pointing normal
                G_new[i, j] = (
                    G_new[i, j][1],
                    G_new[i, j][2],
                    G_old[i, j][3] - direction * τ * g_ij[3] - τ * F_ij[3] + τ * G_stab[3], # G_21 selected by n = (0,1)
                    G_old[i, j][4] - direction * τ * g_ij[4] - τ * F_ij[4] + τ * G_stab[4]) # G_22 selected by n = (0,1)
            end
        elseif id == periodic
            # do nothing
        else
            @assert(id == do_nothing, "not implemented")
        end
    end

    if discretization.filter_mean_G
        mean_G = (mean(getfield.(G, 1)), mean(getfield.(G, 2)), mean(getfield.(G, 3)), mean(getfield.(G, 4)))
        for i in 1:M
            for j in 1:N
                G_new[i, j] = G_new[i, j] .- mean_G
            end
        end
    end

    return τ
end

step (generic function with 1 method)

# Nonlinear and linear material model:

In [12]:
# perform a matrix multiplication between two 4 tuples:

function multiply_tuples(A, B)
    return (
        A[1] * B[1] + A[2] * B[3],
        A[1] * B[2] + A[2] * B[4],
        A[3] * B[1] + A[4] * B[3],
        A[3] * B[2] + A[4] * B[4]
    )
end

# transpose a matrix given as a tuple:

function transpose_tuple(A)
    return (A[1], A[3], A[2], A[4])
end

# Compute trace of a matrix given as a tuple:

function trace_tuple(A)
    return A[1] + A[4]
end

# Compute (Id + A) * B

function transform_tuple(A, B)
    IdA = (1., 0., 0., 1.) .+ A
    return multiply_tuples(IdA, B)
end

transform_tuple (generic function with 1 method)

In [34]:
μ = 0.5
λ = 0.5

function k_prime_nonlinear(A)
    At = transpose_tuple(A)
    ε = 0.5 .* (A .+ At .+ multiply_tuples(At, A))
    S = 2. * μ .* ε .+ λ * trace_tuple(ε) .* (1., 0., 0., 1.)
    return transform_tuple(A, S)
end

function k_prime_linear(A)
    At = transpose_tuple(A)
    ε = 0.5 .* (A .+ At)
    S = 2. * μ .* ε .+ λ * trace_tuple(ε) .* (1., 0., 0., 1.)
    return S
end

function cfl(h)
    # FIXME: what do we want to do here?
    return 0.2 / sqrt(2. * abs(μ) + abs(λ)) * h
end

cfl (generic function with 1 method)

# Initial values, forcing and non-zero Neumann boundary data:

### Verification by comparing with 1D Gaussian

In [14]:
# Initial conditions and forcing

function U_analytic(t, x)
    direction = (1., 1.) ./ sqrt(2.)
    μ = direction[1] * x[1] + direction[2] * x[2] - 2. * sqrt(2)
    # direction = (0., 1.)
    # μ = direction[1] * x[1] + direction[2] * x[2] - 2.
    return (0.25 * exp(-4. * μ^2)) .* direction
end

function G_analytic(t, x)
    return (0., 0., 0., 0.)
end

function F(t, x)
    return (0., 0., 0., 0.)
end

function g(t, x)
    return (0., 0., 0., 0.)
end

# Computational domain Ω

Ω = ((0.0, 0.0), (4.0, 4.0))
k = 4
h = (2.0^(-k), 2.0^(-k))
N = MN(Ω, h)

left_boundary = do_nothing
right_boundary = do_nothing
top_boundary = do_nothing
bottom_boundary = do_nothing

filter_mean_G = true

stabilization = 16.

discretization = Discretization(Ω, h, N, left_boundary, right_boundary, top_boundary, bottom_boundary, filter_mean_G, stabilization);

# Time discretization

T = 1.0
output_granularity = 0.01
out = 1.0e-6

;

### 2D clamped string with "line mass" on the right boundary

In [32]:
# Initial conditions and forcing

function U_analytic(t, x)
    return (0., 0.)
end

function G_analytic(t, x)
    return (0., 0., 0., 0.)
end

function F(t, x)
    return (0., 0., 0., 0.)
end

function g(t, x)
    gravity = 1.0
    if x[1] > 1.0 - 1.0e-10
        return (gravity * x[2] * (1 - x[2]), 0., 0., 0.)
    else
        return (0., 0., 0., 0.)
    end
end

# Computational domain Ω

Ω = ((0.0, 0.0), (1.0, 1.0))
k = 6
h = (2.0^(-k), 2.0^(-k))
N = MN(Ω, h)

left_boundary = dirichlet_u
right_boundary = dirichlet_g_normal
top_boundary = dirichlet_g_normal
bottom_boundary = dirichlet_g_normal

filter_mean_G = false

stabilization = 64.

discretization = Discretization(Ω, h, N, left_boundary, right_boundary, top_boundary, bottom_boundary, filter_mean_G, stabilization);

# Time discretization

T = 30.000
output_granularity = 0.05

out = 1.0e-6

;

# Timestepping:

In [39]:
@time begin
    t = 0
    
    #
    # Create discretization and populate initial values:
    #
    
    M, N = discretization.N
    U = [U_analytic(t, position(discretization, (i, j))) for j = 1:N for i = 1:M]
    U = reshape(U, M, N)
    Us = (copy(U), copy(U))
    G = [G_analytic(t, position(discretization, (i, j))) for j = 1:N for i = 1:M]
    G = reshape(G, M, N)
    Gs = (copy(G), copy(G))

    cycle = 0
    output_cycle = 0
    print("Discretization with (M, N) = (", M, ", ", N, ")\n")
    while true
        U_old = (cycle % 2 == 0) ? Us[1] : Us[2]
        U_new = (cycle % 2 == 0) ? Us[2] : Us[1]
        G_old = (cycle % 2 == 0) ? Gs[1] : Gs[2]
        G_new = (cycle % 2 == 0) ? Gs[2] : Gs[1]

        if output_cycle * output_granularity <= t
            print("output at time t = ", t, " (cycle = ", cycle, ")\n")

            #plt.figure()
            #imshow(getfield.(U_old, 1))
            #plt.savefig("output-U1-" * lpad(output_cycle,4,"0") * ".png")
            #plt.close()

            #plt.figure()
            #imshow(getfield.(U_old, 2))
            #plt.savefig("output-U2-" * lpad(output_cycle,4,"0") * ".png")
            #plt.close()

            xs = [position(discretization, (i, j))[1] for j = 1:N for i = 1:M] .+ reshape(getfield.(U_old, 1), M * N)
            ys = [position(discretization, (i, j))[2] for j = 1:N for i = 1:M] .+ reshape(getfield.(U_old, 2), M * N)
            cs = reshape(norm.(G_old), M * N)

            plt.figure(figsize=(16, 12))
            plt.xlim([-0.1, 1.5]) # TODO
            plt.ylim([-0.1, 1.1]) # TODO
            scatter(xs, ys, s=120, c=cs, cmap=:magma)
            plt.savefig("output-shape-" * lpad(output_cycle,4,"0") * ".png")
            plt.close()
            
            output_cycle = output_cycle + 1
        end

        if (t >= T)
            break
        end
        
        #τ = step(discretization, cfl, k_prime_nonlinear, F, U_old, G_old, t, U_new, G_new)
        τ = step(discretization, cfl, k_prime_linear, F, U_old, G_old, t, U_new, G_new)
        if(τ < out * T)
            print("We didn't make it.\n")
            break
        end
            
        t = t + τ
        cycle = cycle + 1
    end
end

Discretization with (M, N) = (65, 65)
output at time t = 0 (cycle = 0)
output at time t = 0.05103103630798287 (cycle = 20)
output at time t = 0.10206207261596584 (cycle = 40)
output at time t = 0.1505415571085497 (cycle = 59)
output at time t = 0.2015725934165327 (cycle = 79)
output at time t = 0.25005207790911654 (cycle = 98)
output at time t = 0.301083114217099 (cycle = 118)
output at time t = 0.35211415052508144 (cycle = 138)
output at time t = 0.40059363501766476 (cycle = 157)
output at time t = 0.4516246713256472 (cycle = 177)
output at time t = 0.5001041558182305 (cycle = 196)
output at time t = 0.551135192126213 (cycle = 216)
output at time t = 0.6021662284341954 (cycle = 236)
output at time t = 0.6506457129267788 (cycle = 255)
output at time t = 0.7016767492347612 (cycle = 275)
output at time t = 0.7501562337273445 (cycle = 294)
output at time t = 0.801187270035327 (cycle = 314)
output at time t = 0.8522183063433094 (cycle = 334)
output at time t = 0.9006977908358927 (cycle = 3